In [1]:
# =========================================
# 0. CREATE OUTPUT FOLDER
# =========================================
import os

output_folder = "liveComparison"

# Create folder if it does not exist
if not os.path.exists(output_folder):
    os.makedirs(output_folder)


In [2]:
# =========================================
# 1. IMPORT LIBRARIES
# =========================================
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

from textblob import TextBlob
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

import warnings
warnings.filterwarnings("ignore")


In [3]:

# =========================================
# 2. LOAD LIVE CHAT DATA
# =========================================
live = pd.read_csv("Data/English_version/English_translated_live_chat_data.csv")
df = live

print("Dataset shape:", df.shape)

Dataset shape: (290548, 8)


In [4]:
# =========================================
# 3. TEXT CLEANING FUNCTION
# =========================================
def clean_text(text):
    """
    Clean text by:
    - Converting to lowercase
    - Removing URLs
    - Removing special characters
    - Removing extra spaces
    """
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# Apply cleaning
df['clean'] = df['translated_text'].apply(clean_text)

# Remove empty rows
df = df[df['clean'] != ""]



In [5]:
# =========================================
# 4. SENTIMENT ANALYSIS FUNCTION
# =========================================
def get_sentiment_details(text):
    """
    Returns:
    - polarity (-1 to +1)
    - subjectivity (0 to 1)
    - sentiment label (Positive/Negative/Neutral)
    """
    blob = TextBlob(text)

    polarity = blob.sentiment.polarity
    subjectivity = blob.sentiment.subjectivity

    # Define sentiment based on polarity
    if polarity > 0.1:
        sentiment = "Positive"
    elif polarity < -0.1:
        sentiment = "Negative"
    else:
        sentiment = "Neutral"

    return polarity, subjectivity, sentiment


# Apply sentiment analysis
df[['polarity','subjectivity','sentiment']] = df['clean'].apply(
    lambda x: pd.Series(get_sentiment_details(x))
)



In [6]:
# =========================================
# 5. SENTIMENT SUMMARY (NUMERIC)
# =========================================
# Count sentiment per channel
sent_count = df.groupby(['channel','sentiment']).size().unstack().fillna(0)

# Calculate percentage
sent_percent = sent_count.div(sent_count.sum(axis=1), axis=0) * 100

print("\nSentiment Count:\n", sent_count)
print("\nSentiment Percentage:\n", sent_percent)



Sentiment Count:
 sentiment  Negative  Neutral  Positive
channel                               
Ch-1           9965   154852     19439
Ch-2           4105    91420      8623

Sentiment Percentage:
 sentiment  Negative    Neutral   Positive
channel                                  
Ch-1       5.408236  84.041768  10.549996
Ch-2       3.941506  87.778930   8.279564


In [7]:
# =========================================
# 6. SENTIMENT GRAPH (SAVE)
# =========================================
plt.figure(figsize=(8,5))
sent_count.plot(kind='bar')
plt.title("Sentiment Comparison (Ch-1 vs Ch-2)")
plt.xlabel("Channel")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()

# Save graph
plt.savefig(f"{output_folder}/sentiment_comparison.png")
plt.close()



<Figure size 800x500 with 0 Axes>

In [8]:
# =========================================
# 7. POLARITY DISTRIBUTION GRAPH
# =========================================
plt.figure(figsize=(6,4))
sns.histplot(data=df, x='polarity', hue='channel', kde=True)
plt.title("Polarity Distribution by Channel")

plt.savefig(f"{output_folder}/polarity_distribution.png")
plt.close()


In [9]:
# =========================================
# 8. TOPIC MODELING (LDA)
# =========================================
# Convert text into numerical matrix
vectorizer = CountVectorizer(stop_words='english', min_df=2)
dtm = vectorizer.fit_transform(df['clean'])

# Train LDA model (3 topics)
lda = LatentDirichletAllocation(n_components=3, random_state=42)
lda.fit(dtm)

# Get feature names
feature_names = vectorizer.get_feature_names_out()

print("\nTop Words per Topic:")
for i, topic in enumerate(lda.components_):
    words = [feature_names[j] for j in topic.argsort()[-10:]]
    print(f"Topic {i}: {words}")




Top Words per Topic:
Topic 0: ['hii', 'da', 'ami', 'pink', 'waving', 'na', 'hand', 'floor', 'laughing', 'rolling']
Topic 1: ['nice', 'bolo', 'elbowcough', 'green', 'ghost', 'loudly', 'acho', 'tumi', 'ki', 'loading']
Topic 2: ['blue', 'purple', 'hi', 'tears', 'joy', 'red', 'smiling', 'heart', 'eyes', 'face']


In [10]:
# =========================================
# 9. ASSIGN TOPICS TO EACH TEXT
# =========================================
doc_topics = lda.transform(dtm)
df['Topic_Number'] = np.argmax(doc_topics, axis=1)

# Manual topic labels
topic_labels = {
    0: "Positive Feedback",
    1: "Requests/Suggestions",
    2: "Emotional Response"
}

df['Topic_Label'] = df['Topic_Number'].map(topic_labels)


In [11]:
# =========================================
# 10. TOPIC SUMMARY (NUMERIC)
# =========================================
topic_count = df.groupby(['channel','Topic_Label']).size().unstack().fillna(0)

print("\nTopic Count:\n", topic_count)


# =========================================
# 11. TOPIC GRAPH (SAVE)
# =========================================
plt.figure(figsize=(8,5))
topic_count.plot(kind='bar')
plt.title("Topic Comparison (Ch-1 vs Ch-2)")
plt.xlabel("Channel")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()

plt.savefig(f"{output_folder}/topic_comparison.png")
plt.close()



Topic Count:
 Topic_Label  Emotional Response  Positive Feedback  Requests/Suggestions
channel                                                                 
Ch-1                      58416              74797                 51043
Ch-2                      23546              36073                 44529


<Figure size 800x500 with 0 Axes>

In [12]:
# =========================================
# 12. SAVE ALL OUTPUT FILES
# =========================================
df.to_csv(f"{output_folder}/live_full_analysis.csv", index=False)
sent_count.to_csv(f"{output_folder}/sentiment_count.csv")
sent_percent.to_csv(f"{output_folder}/sentiment_percentage.csv")
topic_count.to_csv(f"{output_folder}/topic_count.csv")

print("\nAll files saved successfully in 'comparison' folder!")


All files saved successfully in 'comparison' folder!
